# Dedicated versus shared berths

Does sharing a queue help when the number and speed of the berths stay the same?

* **Dedicated:** two independent M/M/1 queues. Each arriving vessel is assigned randomly with equal probability to one berth and cannot switch queues. Splitting a Poisson arrival stream this way gives two independent Poisson streams, each with half the total arrival rate.
* **Shared:** one FIFO M/M/2 queue. The next vessel uses whichever berth becomes free.

Both layouts have the same total arrival rate and the same service rate **per berth**. There are no quay-length limits, priorities or reneging. This isolates the benefit of pooling, rather than adding capacity. The dedicated layout is simulated as two separate queues, not as one shared queue with two servers.

In [5]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import openqtsim
from IPython.display import display
from openqtsim.mt_engine import create_tasks, run_tasks, statistics_worker, mmc_waiting_time_factor

print(f'This notebook uses OpenQTSim {openqtsim.__version__}')

This notebook uses OpenQTSim v0.5.1


## 1. Define the experiment

The default traffic is 1.6 vessels/hour and each berth serves 1 vessel/hour: both layouts have 80% utilisation. `nr_arr` is the number of customers **per dedicated berth per replication**. The pooled simulation uses twice that many customers, so each layout sees the same total number of customers per replication.

`nr_replications` controls the number of independent experiments. `seed` is a master seed; separate scenario seeds are split further by `create_tasks` into arrival and service seeds for every task. Re-running unchanged inputs is reproducible.

In [6]:
nr_berths = 2
arrival_rate = 1.6  # total vessels/hour
mu_server = 1.0  # vessels/hour per berth
nr_arr = 50_000
nr_replications = 4
seed = 2024
client = None  # serial execution; optionally supply an existing Dask Client

rho = arrival_rate / (nr_berths * mu_server)
if not 0 < rho < 1 or nr_replications < 2:
    raise ValueError('Use positive stable traffic (0 < rho < 1) and at least two replications.')
print(f'Utilisation in both layouts: {rho:.0%}')
print(f'{nr_berths * nr_arr:,} customers per layout per replication; {nr_replications} replications')

Utilisation in both layouts: 80%
100,000 customers per layout per replication; 4 replications


## 2. Construct and simulate both layouts

**Important:** `create_tasks` takes the **total** service capacity in `mu`, not the per-berth rate. A dedicated queue therefore has `mu=mu_server`; the pooled queue has `mu=nr_berths * mu_server`.

Consecutive groups of two dedicated-queue tasks form one experiment. Their average waiting time is customer-weighted: here each queue serves the same number of customers, so the weights are equal. Simulating the two streams independently is equivalent in distribution to the Poisson splitting described above; these are not paired runs of identical vessel visits.

In [7]:
scenario_seeds = [int(child.generate_state(1)[0]) for child in np.random.SeedSequence(seed).spawn(2)]
dedicated_tasks = create_tasks(
    'M', 'M', [1], [rho], nr_arr, mu=mu_server,
    nr_replications=nr_berths * nr_replications, seed=scenario_seeds[0],
)
pooled_tasks = create_tasks(
    'M', 'M', [nr_berths], [rho], nr_berths * nr_arr, mu=nr_berths * mu_server,
    nr_replications=nr_replications, seed=scenario_seeds[1],
)
dedicated_results = pd.DataFrame(run_tasks(dedicated_tasks, client, func=statistics_worker))
pooled_results = pd.DataFrame(run_tasks(pooled_tasks, client, func=statistics_worker))
dedicated_results['experiment'] = dedicated_results['replication'] // nr_berths
replications = pd.DataFrame({
    'Dedicated': dedicated_results.groupby('experiment')['W_q'].mean(),
    'Shared': pooled_results.set_index('replication')['W_q'],
})
replications.index.name = 'replication'
display(replications.rename_axis(columns='Layout').round(4))

Layout,Dedicated,Shared
replication,,
0,3.9341,1.7869
1,4.0977,1.8318
2,3.8422,1.7670
3,3.9186,1.8553


## 3. Compare with steady-state queueing theory

For each dedicated berth, $W_q = \rho/[\mu_{server}(1-\rho)]$. For the shared queue, Erlang C gives the waiting-time factor; dividing by the per-berth service rate converts it to hours.

The error bars show **one standard error of the mean**, calculated from independent experiments, not the spread of individual vessel waiting times. Every simulation starts empty and includes final draining. These finite-run estimates have start-up effects; error bars quantify replication variability, not start-up bias. Increase both run length and replications to investigate convergence, especially near full occupancy.

In [8]:
comparison = pd.DataFrame({
    'analytical W_q (hours)': [
        mmc_waiting_time_factor(1, rho) / mu_server,
        mmc_waiting_time_factor(nr_berths, rho) / mu_server,
    ],
    'simulated W_q (hours)': replications.mean(),
    'standard error (hours)': replications.std(ddof=1) / np.sqrt(nr_replications),
}, index=['Dedicated', 'Shared'])
display(comparison.round(4))
reduction = 1 - comparison.loc['Shared', 'analytical W_q (hours)'] / comparison.loc['Dedicated', 'analytical W_q (hours)']
print(f'Analytical reduction in mean waiting time from pooling: {reduction:.1%}')

fig = go.Figure()
fig.add_bar(x=comparison.index, y=comparison['simulated W_q (hours)'], name='Simulation (mean +/- 1 SE)',
            error_y=dict(type='data', array=comparison['standard error (hours)']))
fig.add_scatter(x=comparison.index, y=comparison['analytical W_q (hours)'], mode='markers',
                marker=dict(color='black', symbol='diamond', size=12), name='Analytical steady state')
fig.update_layout(title=f'Same traffic, same {nr_berths} berths, different queue organisation',
                  yaxis_title='Mean waiting time W_q (hours)', template='plotly_white', height=500)
fig.show()

,analytical W_q (hours),simulated W_q (hours),standard error (hours)
Dedicated,4.0000,3.9482,0.0537
Shared,1.7778,1.8103,0.0202


Analytical reduction in mean waiting time from pooling: 55.6%


## 4. What does pooling change?

With the default inputs, theory gives **4.000 hours** waiting for dedicated berths and **1.778 hours** for shared berths: a **55.6% reduction without extra capacity**. A dedicated berth can be idle while vessels wait at the other berth. Pooling avoids that imbalance, although it does not remove waiting altogether.

Try a lower arrival rate, or increase `nr_berths` while keeping the utilisation comparable. Real pooling benefits may be limited by berth compatibility, contractual assignments, vessel size or handling equipment; those restrictions are intentionally absent here.

For adding physical capacity at fixed traffic, see **3003 - Berth planning by cost**. For the effect of occupancy, see **3002 - Berth occupancy and waiting time**.